In [ ]:
import json
from datetime import datetime

# Clase Dirección (Composición)
class Direccion:
    def __init__(self, calle, ciudad, codigo_postal):
        self.calle = calle
        self.ciudad = ciudad
        self.codigo_postal = codigo_postal

    def __str__(self):
        return f"{self.calle}, {self.ciudad}, {self.codigo_postal}"

# Clase Cliente
class Cliente:
    def __init__(self, nombre, telefono, email, direccion):
        self.nombre = nombre
        self.__telefono = telefono
        self.__email = email
        self.direccion = direccion

    def get_telefono(self):
        return self.__telefono

    def set_telefono(self, telefono):
        self.__telefono = telefono

    def get_email(self):
        return self.__email

    def set_email(self, email):
        self.__email = email

    def __str__(self):
        return f"Cliente: {self.nombre}, Teléfono: {self.__telefono}, Email: {self.__email}, Dirección: {self.direccion}"

# Clase Paquete (Base)
class Paquete:
    def __init__(self, peso, dimensiones, tipo, precio_base):
        self.__peso = peso
        self.__dimensiones = dimensiones
        self.__tipo = tipo
        self.__precio_base = precio_base

    def get_peso(self):
        return self.__peso

    def set_peso(self, peso):
        if peso > 0:
            self.__peso = peso
        else:
            raise ValueError("El peso debe ser positivo")

    def get_dimensiones(self):
        return self.__dimensiones

    def calcular_precio(self):
        return self.__precio_base

    def __str__(self):
        return f"Paquete: {self.__tipo}, Peso: {self.__peso}kg, Dimensiones: {self.__dimensiones}"

# Clase PaqueteExpress (Hereda de Paquete)
class PaqueteExpress(Paquete):
    def __init__(self, peso, dimensiones, precio_base, recargo):
        super().__init__(peso, dimensiones, "Express", precio_base)
        self.__recargo = recargo

    def calcular_precio(self):
        return super().calcular_precio() + self.__recargo

    def __str__(self):
        return f"Paquete Express: {self.get_peso()}kg, Dimensiones: {self.get_dimensiones()}, Precio: {self.calcular_precio()}"

# Clase PaqueteEstandar (Hereda de Paquete)
class PaqueteEstandar(Paquete):
    def __init__(self, peso, dimensiones, precio_base):
        super().__init__(peso, dimensiones, "Estandar", precio_base)

    def __str__(self):
        return f"Paquete Estándar: {self.get_peso()}kg, Dimensiones: {self.get_dimensiones()}, Precio: {self.calcular_precio()}"

# Clase Envio
class Envio:
    def __init__(self, paquete, cliente):
        self.paquete = paquete
        self.cliente = cliente
        self.estado = "Pendiente"
        self.fecha_envio = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def set_estado(self, estado):
        if estado in ["Pendiente", "En tránsito", "Entregado"]:
            self.estado = estado
        else:
            raise ValueError("Estado inválido")

    def __str__(self):
        return f"Envio a {self.cliente.nombre}, {self.paquete}, Estado: {self.estado}, Fecha: {self.fecha_envio}"

# Clase SistemaEnvios
class SistemaEnvios:
    def __init__(self):
        self.clientes = []
        self.envios = []

    def registrar_cliente(self, cliente):
        self.clientes.append(cliente)

    def registrar_envio(self, envio):
        self.envios.append(envio)

    def listar_envios(self):
        for envio in self.envios:
            print(envio)

    def calcular_totales(self):
        total_express = sum([envio.paquete.calcular_precio() for envio in self.envios if isinstance(envio.paquete, PaqueteExpress)])
        total_estandar = sum([envio.paquete.calcular_precio() for envio in self.envios if isinstance(envio.paquete, PaqueteEstandar)])
        print(f"Total Envíos Express: {total_express}")
        print(f"Total Envíos Estándar: {total_estandar}")

    def guardar_datos(self, filename):
        data = {
            "clientes": [{"nombre": cliente.nombre, "telefono": cliente.get_telefono(), "email": cliente.get_email(), "direccion": str(cliente.direccion)} for cliente in self.clientes],
            "envios": [{"cliente": envio.cliente.nombre, "paquete": str(envio.paquete), "estado": envio.estado, "fecha_envio": envio.fecha_envio} for envio in self.envios]
        }
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)

    def cargar_datos(self, filename):
        try:
            with open(filename, 'r') as f:
                data = json.load(f)
                for cliente_data in data["clientes"]:
                    direccion = Direccion(cliente_data["direccion"].split(", ")[0], cliente_data["direccion"].split(", ")[1], cliente_data["direccion"].split(", ")[2])
                    cliente = Cliente(cliente_data["nombre"], cliente_data["telefono"], cliente_data["email"], direccion)
                    self.registrar_cliente(cliente)
                
                for envio_data in data["envios"]:
                    cliente = next(cliente for cliente in self.clientes if cliente.nombre == envio_data["cliente"])
                    # Crear el paquete según el tipo
                    if "Express" in envio_data["paquete"]:
                        paquete_data = envio_data["paquete"].split(", ")
                        peso = float(paquete_data[1].split(":")[1].replace("kg", "").strip())
                        dimensiones = paquete_data[2].split(":")[1].strip()
                        precio = float(paquete_data[3].split(":")[1].strip())
                        recargo = float(paquete_data[4].split(":")[1].strip())
                        paquete = PaqueteExpress(peso, dimensiones, precio, recargo)
                    else:
                        paquete_data = envio_data["paquete"].split(", ")
                        peso = float(paquete_data[1].split(":")[1].replace("kg", "").strip())
                        dimensiones = paquete_data[2].split(":")[1].strip()
                        precio = float(paquete_data[3].split(":")[1].strip())
                        paquete = PaqueteEstandar(peso, dimensiones, precio)

                    envio = Envio(paquete, cliente)
                    self.registrar_envio(envio)
        except FileNotFoundError:
            print("El archivo no existe.")
        except KeyError as e:
            print(f"Error al procesar los datos: falta la clave {e}.")

# Menú de opciones
def menu():
    sistema = SistemaEnvios()

    while True:
        print("\n--- Sistema de Gestión de Paquetería ---")
        print("1. Registrar Cliente")
        print("2. Registrar Envío")
        print("3. Listar Envíos")
        print("4. Ver Totales por Tipo de Envío")
        print("5. Guardar Datos")
        print("6. Cargar Datos")
        print("7. Editar Cliente")
        print("8. Salir")

        opcion = input("Selecciona una opción: ")

        if opcion == '1':
            nombre = input("Nombre del Cliente: ")
            telefono = input("Teléfono del Cliente: ")
            email = input("Email del Cliente: ")
            calle = input("Calle: ")
            ciudad = input("Ciudad: ")
            codigo_postal = input("Código Postal: ")
            direccion = Direccion(calle, ciudad, codigo_postal)
            cliente = Cliente(nombre, telefono, email, direccion)
            sistema.registrar_cliente(cliente)

        elif opcion == '2':
            tipo = input("Tipo de Paquete (Express/Estandar): ")
            peso = float(input("Peso del Paquete (kg): "))
            dimensiones = input("Dimensiones del Paquete (LxWxH): ")
            precio_base = float(input("Precio Base del Paquete: "))
            
            if tipo.lower() == "express":
                recargo = float(input("Recargo Express: "))
                paquete = PaqueteExpress(peso, dimensiones, precio_base, recargo)
            else:
                paquete = PaqueteEstandar(peso, dimensiones, precio_base)
            
            cliente_nombre = input("Nombre del Cliente para el Envío: ")
            cliente = next(cliente for cliente in sistema.clientes if cliente.nombre == cliente_nombre)
            envio = Envio(paquete, cliente)
            sistema.registrar_envio(envio)

        elif opcion == '3':
            sistema.listar_envios()

        elif opcion == '4':
            sistema.calcular_totales()

        elif opcion == '5':
            archivo = input("Nombre del archivo para guardar: ")
            sistema.guardar_datos(archivo)

        elif opcion == '6':
            archivo = input("Nombre del archivo para cargar: ")
            sistema.cargar_datos(archivo)

        elif opcion == '7':
            cliente_nombre = input("Nombre del Cliente a editar: ")
            cliente = next(cliente for cliente in sistema.clientes if cliente.nombre == cliente_nombre)
            telefono = input(f"Nuevo Teléfono (actual: {cliente.get_telefono()}): ")
            email = input(f"Nuevo Email (actual: {cliente.get_email()}): ")
            cliente.set_telefono(telefono)
            cliente.set_email(email)
            print("Datos actualizados correctamente.")

        elif opcion == '8':
            print("¡Hasta luego!")
            break

        else:
            print("Opción no válida. Intenta de nuevo.")

# Llamada al menú
if __name__ == '__main__':
    menu()



1. Registrar cliente
2. Registrar envío
3. Mostrar envíos
4. Generar reporte
5. Guardar datos
6. Mostrar totales de ventas
7. Salir
